In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_pretrain_MOE_collapse_HPO_DeepCNNLSTM"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_pretrain_MOE_collapse_HPO_DeepCNNLSTM
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\1s3w_pretrain_MOE_collapse_HPO_DeepCNNLSTM.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'pretrain_hpo_DeepCNNLSTM_moe'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 1000 trials.
Best value (Accuracy): 0.3625
Best Hyperparameters:
  learning_rate: 0.0002557425603318319
  weight_decay: 0.004015827534295605
  dropout: 0.4
  label_smooth: 0.05
  batch_size: 64
  optimizer: adamw
  cnn_base_filters: 32
  lstm_hidden: 128
  bidirectional: True
  head_type: linear
  MOE_placement: encoder
  MOE_gate_temperature: 2.076814415679466
  MOE_aux_coeff: 0.023762197321851686
  MOE_ctx_out_dim: 64
  MOE_ctx_hidden_dim: 128
  MOE_dropout: 0.09428864398511298
  MOE_expert_expand: 1.183139297688831
  MOE_mlp_hidden_mult: 1.1118495038239677


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL
params_to_plot = ["learning_rate", "weight_decay", "dropout", "label_smooth", "batch_size", "optimizer", "cnn_base_filters", "lstm_hidden", "bidirectional", "head_type", 
                  "MOE_placement", "MOE_gate_temperature", "MOE_aux_coeff", "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout", "MOE_expert_expand", "MOE_mlp_hidden_mult"]

# OPTIMIZATION
params_to_plot_OPT = ["learning_rate", "weight_decay", "dropout", "label_smooth", "batch_size", "optimizer"]

# ARCHITECTURE
params_to_plot_ARCH = ["cnn_base_filters", "lstm_hidden", "bidirectional", "head_type"]

# MOE AUX LOSS
params_to_plot_MOE_AUX = ["MOE_gate_temperature", "MOE_aux_coeff", "MOE_dropout"]

# MOE MISC
params_to_plot_MOE_MISC = ["MOE_placement", "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_expert_expand", "MOE_mlp_hidden_mult"]


In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_ARCH)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_AUX)
fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_MISC)
fig_slice.show()

In [14]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
867,867,0.3625,None,2026-04-02 18:40:43.530053,0 days 00:00:20.536266
931,931,0.3625,None,2026-04-02 18:44:05.320526,0 days 00:00:10.270661
139,139,0.3625,None,2026-04-02 18:11:38.603399,0 days 00:00:11.914616
559,559,0.3625,None,2026-04-02 18:28:09.112268,0 days 00:00:11.860712
727,727,0.3625,None,2026-04-02 18:34:44.137675,0 days 00:00:09.103195
983,983,0.3625,None,2026-04-02 18:46:48.367589,0 days 00:00:23.455237
349,349,0.3625,None,2026-04-02 18:19:38.121171,0 days 00:00:09.603949
131,131,0.3500,None,2026-04-02 18:11:26.829206,0 days 00:00:09.708049
793,793,0.3500,None,2026-04-02 18:37:34.254848,0 days 00:00:11.539635
864,864,0.3500,None,2026-04-02 18:40:42.231512,0 days 00:00:22.852296
